In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # ARC LAB — Bronze Ingest (Real-Time)
# MAGIC **Pulls the current carbon intensity signal from WattTime and appends to Bronze.**
# MAGIC
# MAGIC - Region: CAISO_NORTH
# MAGIC - Signal: co2_moer (lbs CO2 per MWh)
# MAGIC - Endpoint: /v3/signal-index (real-time)
# MAGIC - Mode: append-only

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Config

# COMMAND ----------

import requests
from datetime import datetime, timezone, date
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, DateType, TimestampNTZType
)

CATALOG      = "bootcamp_students"
BRONZE_TABLE = f"{CATALOG}.bronze.watttime_raw"
REGION       = "CAISO_NORTH"
SIGNAL_TYPE  = "co2_moer"
API_BASE     = "https://api.watttime.org/v3"

print(f"Target table : {BRONZE_TABLE}")
print(f"Region       : {REGION}")
print(f"Signal       : {SIGNAL_TYPE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Authenticate

# COMMAND ----------

USERNAME = dbutils.secrets.get(scope="arc-lab", key="watttime-username")
PASSWORD = dbutils.secrets.get(scope="arc-lab", key="watttime-password")

def get_watttime_token(username, password):
    response = requests.get(
        "https://api.watttime.org/login",
        auth=(username, password),
        timeout=10
    )
    response.raise_for_status()
    return response.json()["token"]

token = get_watttime_token(USERNAME, PASSWORD)
print(f"✓ Token acquired at {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Call WattTime API

# COMMAND ----------

url     = f"{API_BASE}/signal-index"
headers = {"Authorization": f"Bearer {token}"}
params  = {
    "region":      REGION,
    "signal_type": SIGNAL_TYPE
}

response = requests.get(url, headers=headers, params=params, timeout=15)
response.raise_for_status()
raw = response.json()

print(f"✓ API response received")
print(f"  model_version : {raw['meta']['model']['date']}")
print(f"  units         : {raw['meta']['units']}")
print(f"  records       : {len(raw['data'])}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Parse Response

# COMMAND ----------

ingested_at    = datetime.now(timezone.utc).replace(tzinfo=None)
ingestion_date = ingested_at.date()
model_version  = raw["meta"]["model"]["date"]
units          = raw["meta"]["units"]

rows = []
for record in raw["data"]:
    ts = datetime.fromisoformat(record["point_time"]).replace(tzinfo=None)
    rows.append((
        REGION,
        SIGNAL_TYPE,
        ts,
        float(record["value"]),
        units,
        model_version,
        "/v3/signal-index",
        ingested_at,
        ingestion_date,
        "realtime"
    ))

print(f"✓ Parsed {len(rows)} record(s)")
print(f"  ts_utc : {rows[0][2]}")
print(f"  value  : {rows[0][3]} {units}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Write to Bronze

# COMMAND ----------

bronze_schema = StructType([
    StructField("region_code",    StringType(),       True),
    StructField("signal_type",    StringType(),       True),
    StructField("ts_utc",         TimestampNTZType(), True),
    StructField("value",          DoubleType(),       True),
    StructField("units",          StringType(),       True),
    StructField("model_version",  StringType(),       True),
    StructField("api_endpoint",   StringType(),       True),
    StructField("ingested_at",    TimestampNTZType(), True),
    StructField("ingestion_date", DateType(),         True),
    StructField("source_file",    StringType(),       True),
])

df = spark.createDataFrame(rows, schema=bronze_schema)

df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(BRONZE_TABLE)

print(f"✓ Written {len(rows)} record(s) to {BRONZE_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Verify

# COMMAND ----------

display(spark.sql(f"""
    SELECT source_file, units, COUNT(*) as record_count,
           MIN(ts_utc) as earliest, MAX(ts_utc) as latest
    FROM {BRONZE_TABLE}
    GROUP BY source_file, units
    ORDER BY source_file
"""))

Target table : bootcamp_students.bronze.watttime_raw
Region       : CAISO_NORTH
Signal       : co2_moer
✓ Token acquired at 2026-03-08 05:21:47 UTC
✓ API response received
  model_version : 2023-03-01
  units         : percentile
  records       : 1
✓ Parsed 1 record(s)
  ts_utc : 2026-03-08 05:20:00
  value  : 99.0 percentile
✓ Written 1 record(s) to bootcamp_students.bronze.watttime_raw


source_file,units,record_count,earliest,latest
null,percentile,1,2026-03-06T23:05:00.000,2026-03-06T23:05:00.000
backfill_7day,lbs_co2_per_mwh,2016,2026-02-27T23:20:00.000,2026-03-06T23:15:00.000
realtime,percentile,1,2026-03-08T05:20:00.000,2026-03-08T05:20:00.000
